[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境检查与全课数据管线

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 跑通环境，并学会本课**所有 notebook 都复用**的真实数据下载缓存范式。

**本 notebook 你将：**
1. 实现一个"下载一次、缓存复用"的 `load_csv` 工具
2. 加载真实的 **Palmer Penguins** 数据集，亲眼看到真实数据的"脏"（缺失值、量纲差异、类别不平衡）
3. 手写**标准化**与**分层划分（stratified split）**
4. 用 **majority-class baseline** 理解为什么 accuracy 会骗人

> 数据来源：`seaborn-data`（GitHub raw）。首次运行联网下载并缓存到 `~/.ml_foundations_data/`。

## 1 · 全课复用的数据管线

下面这个 `load_csv` 是本课每个 notebook 的第一块代码。它做三件事：确保缓存目录存在、首次下载、之后直接读本地。
**下载一次、缓存复用、可重复** —— 这正是真实评测里管理数据集应有的样子。

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd

CACHE = os.path.expanduser("~/.ml_foundations_data")
os.makedirs(CACHE, exist_ok=True)

def fetch(url, fname):
    "下载 url 到缓存（若不存在），返回本地路径。"
    path = os.path.join(CACHE, fname)
    if not os.path.exists(path):
        print(f"下载 {fname} ...")
        urllib.request.urlretrieve(url, path)
    return path

def load_csv(url, fname, **read_kw):
    "下载并用 pandas 读成 DataFrame。"
    return pd.read_csv(fetch(url, fname), **read_kw)

np.set_printoptions(precision=4, suppress=True)
print("缓存目录:", CACHE)

## 2 · 加载真实数据：Palmer Penguins

344 只企鹅，4 个数值测量（喙长、喙深、鳍长、体重）+ 物种/岛屿/性别。
注意真实数据的三个特征：**有缺失值、特征量纲差很多、类别不完全均衡**。

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = load_csv(URL, "penguins.csv")

print("形状:", df.shape)
print("\n列:", list(df.columns))
print("\n每列缺失值数:")
print(df.isna().sum())
print("\n物种分布（类别不平衡）:")
print(df["species"].value_counts())

num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print("\n数值特征的量纲（min~max）—— 体重比喙深大几百倍:")
print(df[num_cols].describe().loc[["min", "max"]].T)

## 3 · 清洗缺失 + 标准化

体重以克计（~3000–6000），喙深以毫米计（~13–22）。若不标准化直接喂给梯度下降或距离算法，
**体重会因为数值大而主导一切**。标准化 = 减均值除标准差，让每个特征同量纲、均值 0 方差 1。

In [ ]:
clean = df.dropna(subset=num_cols + ["species"]).reset_index(drop=True)
print(f"丢弃缺失后: {len(df)} -> {len(clean)} 行")

X = clean[num_cols].to_numpy(dtype=float)
mu, sd = X.mean(axis=0), X.std(axis=0)
Xs = (X - mu) / sd

print("\n标准化前 均值:", X.mean(0))
print("标准化前 标准差:", X.std(0))
print("\n标准化后 均值:", Xs.mean(0).round(6), "(≈0)")
print("标准化后 标准差:", Xs.std(0).round(6), "(≈1)")

## 4 · 分层划分 + majority baseline

随机划分可能让某个稀有类在训练集里几乎消失。**分层划分**保证每个类在 train/test 里比例一致。
然后我们算 **majority-class baseline**：永远预测最多的那一类能拿多少 accuracy —— 任何模型都得先打败它。

In [ ]:
# 物种 -> 整数标签
species = clean["species"].to_numpy()
classes, y = np.unique(species, return_inverse=True)
print("类别:", dict(enumerate(classes)))

# 简单的分层划分（演示版）
rng = np.random.default_rng(0)
train_idx, test_idx = [], []
for c in np.unique(y):
    idx = np.where(y == c)[0]
    rng.shuffle(idx)
    cut = int(0.7 * len(idx))
    train_idx += list(idx[:cut]); test_idx += list(idx[cut:])
train_idx, test_idx = np.array(train_idx), np.array(test_idx)

print(f"\ntrain={len(train_idx)} test={len(test_idx)}")
print("train 各类占比:", np.bincount(y[train_idx]) / len(train_idx))
print("test  各类占比:", np.bincount(y[test_idx]) / len(test_idx), "（与 train 一致 = 分层成功）")

# majority baseline
maj = np.bincount(y[train_idx]).argmax()
baseline_acc = (y[test_idx] == maj).mean()
print(f"\nmajority-class baseline accuracy = {baseline_acc:.3f}")
print("=> 任何模型 accuracy 不超过这个数，就等于什么都没学到。")

---
## ✏️ 练习区

下面 4 道题给 `TODO` 骨架与 `assert` 自测。**先自己写，全过再看文末参考答案。**

### ✏️ 练习 1：带缓存的下载器

实现 `download_cached(url, fname)`：若缓存里已有 `fname` 就直接返回路径、**不重复下载**；否则下载。
返回本地路径字符串。

In [ ]:
def download_cached(url, fname):
    # TODO: 用 os.path.join(CACHE, fname)；os.path.exists 判断；
    #       不存在则 urllib.request.urlretrieve(url, path)；返回 path
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
p = download_cached(URL, "penguins.csv")
assert os.path.exists(p) and p.endswith("penguins.csv")
import time; t0 = time.time(); download_cached(URL, "penguins.csv");
assert time.time() - t0 < 1.0, "第二次应命中缓存、秒回"
print("练习 1 通过 ✓")


### ✏️ 练习 2：在 train 上 fit、在 test 上 apply 的标准化

**陷阱**：标准化的均值/标准差只能用**训练集**算，再套到测试集——否则就是数据泄漏。
实现 `standardize_fit_apply(X_train, X_test)`，返回 `(Xtr_std, Xte_std, mu, sd)`。

In [ ]:
def standardize_fit_apply(X_train, X_test):
    # TODO: mu, sd 用 X_train 算；两个集合都用同一组 mu/sd 标准化
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
Xtr, Xte = X[train_idx], X[test_idx]
a, b, m, s = standardize_fit_apply(Xtr, Xte)
assert np.allclose(a.mean(0), 0, atol=1e-9) and np.allclose(a.std(0), 1, atol=1e-9)
assert np.allclose(m, Xtr.mean(0)) and np.allclose(s, Xtr.std(0))
# 测试集不一定均值0方差1（因为用的是 train 的统计量）—— 这才正确
assert not np.allclose(b.mean(0), 0, atol=1e-3)
print("练习 2 通过 ✓")


### ✏️ 练习 3：分层划分

实现 `stratified_split(y, frac, seed)`，返回 `(train_idx, test_idx)`，
使**每个类**约 `frac` 进训练集。要求结果可复现（用给定 seed）。

In [ ]:
def stratified_split(y, frac=0.7, seed=0):
    # TODO: 对每个类各自 shuffle 后按 frac 切，合并各类的 train/test 索引
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
tr, te = stratified_split(y, 0.7, seed=1)
assert len(set(tr) & set(te)) == 0, "train/test 不能重叠"
assert len(tr) + len(te) == len(y)
p_tr = np.bincount(y[tr]) / len(tr)
p_all = np.bincount(y) / len(y)
assert np.allclose(p_tr, p_all, atol=0.05), "各类比例应与全集接近"
print("练习 3 通过 ✓")


### ✏️ 练习 4：majority baseline 与不平衡陷阱

实现 `majority_baseline(y_train, y_test)`，返回在测试集上"永远猜训练集最多类"的 accuracy。
然后回答：若某数据集 95% 是 0 类，baseline 是多少？（写进 `answer_q4`）

In [ ]:
def majority_baseline(y_train, y_test):
    # TODO: 训练集最多的类 -> 在测试集上全猜它的 accuracy
    raise NotImplementedError

answer_q4 = None  # TODO: 填 95% 为 0 类时的 baseline accuracy（小数）


In [ ]:
# —— 练习 4 自测 ——
acc = majority_baseline(y[train_idx], y[test_idx])
assert abs(acc - baseline_acc) < 1e-9
assert abs(answer_q4 - 0.95) < 1e-9, "95% 是 0 类 => baseline=0.95，所以 95% 的 accuracy 可能毫无意义"
print(f"练习 4 通过 ✓  baseline={acc:.3f}")


---
## 📖 参考答案

先自己完成上面的 TODO，再展开核对。

In [ ]:
# 练习 1 参考答案
def download_cached(url, fname):
    path = os.path.join(CACHE, fname)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path
p = download_cached(URL, "penguins.csv")
assert os.path.exists(p) and p.endswith("penguins.csv")
print("练习 1 ✓")

In [ ]:
# 练习 2 参考答案
def standardize_fit_apply(X_train, X_test):
    mu = X_train.mean(axis=0); sd = X_train.std(axis=0)
    return (X_train - mu) / sd, (X_test - mu) / sd, mu, sd
Xtr, Xte = X[train_idx], X[test_idx]
a, b, m, s = standardize_fit_apply(Xtr, Xte)
assert np.allclose(a.mean(0), 0, atol=1e-9) and np.allclose(m, Xtr.mean(0))
print("练习 2 ✓")

In [ ]:
# 练习 3 参考答案
def stratified_split(y, frac=0.7, seed=0):
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]; rng.shuffle(idx)
        cut = int(round(frac * len(idx)))
        tr += list(idx[:cut]); te += list(idx[cut:])
    return np.array(tr), np.array(te)
tr, te = stratified_split(y, 0.7, seed=1)
assert len(set(tr) & set(te)) == 0 and len(tr) + len(te) == len(y)
print("练习 3 ✓")

In [ ]:
# 练习 4 参考答案
def majority_baseline(y_train, y_test):
    maj = np.bincount(y_train).argmax()
    return (y_test == maj).mean()
answer_q4 = 0.95
assert abs(majority_baseline(y[train_idx], y[test_idx]) - baseline_acc) < 1e-9
print("练习 4 ✓  —— 不平衡数据上，先报 baseline 再报模型分，是评测科学家的肌肉记忆")